Q1. What are the most common reasons for missing data in ETL pipelines?

-> Missing data usually happens because of issues at source, during extraction, or during transformation.

Common reasons:
1. Data not collected → Optional fields (income, phone number) left blank by users
2. System or sensor failures → Logs not generated, API timeouts
3. Data integration issues → LEFT JOINs causing NULLs when no match exists
4. Human error → Manual entry mistakes
5. Schema mismatch → Source column exists but target table doesn’t map it
6. Data corruption or truncation → File transfer or encoding problems


Q2. Why is blindly deleting rows with missing values considered a bad practice in ETL?


Because we might: Lose valuable information
                               Reduce dataset size significantly
                               Introduce bias
Example: If 30% of customers don’t report income and we delete them → our analysis now represents only high-income or willing users, not all customers.


Q3. Difference between Listwise deletion and Column deletion

1. Listwise Deletion ➡ Remove entire rows that contain any missing value.
When appropriate:
          Very small amount of missing data (e.g., <2%)
          Data is missing completely at random (MCAR)
          Dataset is large
Example:  
          Remove rows where age OR salary is NULL

2. Column Deletion ➡ Remove entire columns that have too many missing values.
When appropriate:
        Column has very high missing rate (e.g., 70–90%)
        Column is not business-critical
Example:
        Drop “secondary_phone_number” column if 85% values are NULL.


Q4. Why is median imputation preferred over mean for skewed data (e.g., income)?
Because mean is sensitive to outliers, median is not.

Example (Income):
             20000, 25000, 30000, 50000, 10,000,000

Mean → very high (distorted)
Median → realistic middle value


Q5. What is forward fill and where is it most useful?

Forward fill (ffill): ➡ Fill missing value using the previous known value
Most useful in:
        Time-series data
        Sensor data
        Stock prices
        Daily logs


Q6. Why flag missing values before imputation in ETL?

Because once you impute, you lose information that the value was missing.

 Best practice:
                   Create a flag column
                   income_missing_flag = 1/0

Why important: Models can learn patterns from missingness
Helps in debugging data quality
Supports audit & explainability


Q7. How can missing income itself provide business insights?

Missing income is not random.
Possible insights:
      High-value customers may avoid sharing income (privacy concerns)
       Certain regions or age groups skip income fields

Missing income might correlate with:
       Higher churn
       Lower engagement
       Specific marketing responses

Business use:
    Segment customers who don’t disclose income
    Adjust marketing or pricing strategies
    Improve data collection forms



In [ ]:
CREATE DATABASE assignment;
USE assignment;

CREATE TABLE customers(
    customer_id INT PRIMARY KEY,
    name VARCHAR(50),
    city VARCHAR(50),
    monthly_sales INT,
    income INT,
    region VARCHAR(20)
    );

INSERT INTO customers
(customer_id, name, city, monthly_sales, income, region)
VALUES
(101, 'Rahul Mehta', 'Mumbai', 12000, 65000, 'West'),
(102, 'Anjali Rao', 'Bengaluru', NULL, NULL, 'South'),
(103, 'Suresh Iyer', 'Chennai', 15000, 72000, 'South'),
(104, 'Neha Singh', 'Delhi', NULL, NULL, 'North'),
(105, 'Amit Verma', 'Pune', 18000, 58000, NULL),
(106, 'Karan Shah', 'Ahmedabad', NULL, 61000, 'West'),
(107, 'Pooja Das', 'Kolkata', 14000, NULL, 'East'),
(108, 'Riya Kapoor', 'Jaipur', 16000, 69000, 'North');


SELECT * FROM customers;



-- Q8. Listwise Deletion
-- Remove all rows where Region is missing

-- Identify affected rows

SELECT *
FROM customers
WHERE region IS NULL;

-- Show the dataset after deletion
SET SQL_SAFE_UPDATES = 0;
DELETE
FROM customers
WHERE region IS NULL;

SET SQL_SAFE_UPDATES = 1;
-- Mention how many records were lost
SELECT COUNT(*) AS total_before
FROM customers;

-- here we can find only 1 record lost

--
--  Q9. Imputation

-- Handle missing values in Monthly_Sales using:
-- Forward Fill

SELECT customer_id, monthly_sales
FROM customers
ORDER BY customer_id;



-- Apply forward fill
-- Show before vs after values


SELECT
    customer_id,
    monthly_sales,
    COALESCE(
        monthly_sales,
        (
            SELECT c2.monthly_sales
            FROM customers c2
            WHERE c2.customer_id < c.customer_id
              AND c2.monthly_sales IS NOT NULL
            ORDER BY c2.customer_id DESC
            LIMIT 1
        )
    ) AS monthly_sales_filled
FROM customers c
ORDER BY customer_id;



-- Explain why forward fill is suitable here

--  Forward fill is suitable because Monthly_Sales is a sequential and continuous business metric,
-- and using the previous valid value preserves sales trends without introducing artificial averages or losing data.

-- Q10. Flagging Missing Data

-- Create a flag column for missing Income.

ALTER TABLE customers
ADD income_missing_flag INT;

-- Tasks:
-- Create Income_Missing_Flag (0 = present, 1 = missing)
SET SQL_SAFE_UPDATES = 0;
UPDATE customers
SET income_missing_flag =
    CASE
        WHEN income IS NULL THEN 1
        ELSE 0
    END;
SET SQL_SAFE_UPDATES = 1;
-- Show updated dataset
SELECT
    customer_id,
    name,
    income,
    income_missing_flag
FROM customers;

-- Count how many customers have missing income
SELECT COUNT(*) AS missing_income_count
FROM customers
WHERE income_missing_flag = 1;